In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import scipy as sp
from scipy import stats
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# ??????????kaggle??????,???????
import sys
print(sys.version)
print(np.__version__)
print(pd.__version__)
print(sp.__version__)

# **????**

In [ ]:
df_train = pd.read_csv("../input/train.csv")
df_test = pd.read_csv("../input/test.csv")
df_allX = pd.concat([df_train.loc[:,'MSSubClass':'SaleCondition'],
                   df_test.loc[:,'MSSubClass':'SaleCondition']])
df_allX = df_allX.reset_index(drop=True)

**Tips:**  
read_csv() ????? NA:

- `pd.read_csv(file,... na_values=None, keep_default_na=True, ...)`
- na_values: ???????????,???? np.NaN(float?),????????orobject??
    - ???:`'', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan','1.#IND', '1.#QNAN', 'N/A', 'NA', 'NULL', 'NaN', 'nan'`.
- keep_default_na
    - True:?csv ??????or???? na_values ? default ?????,?????? np.NaN
    - False:
        - na_values=[...] :????? na_values ??,?????? np.NaN
        - na_values???:????????,???????,???:??????feature????? object ? —— so,???

In [ ]:
print(df_train.shape,df_test.shape,df_allX.shape) # df_allX ?? Id ? SalePrice ??

## ????

In [ ]:
df_train.describe()

## ????

**???2???????:**

| |**???(numeric)**|**???(object)**|
|:---|:---|:---|
|**???(Continuous)**|?????……| ? |
|**???(discrete)<br>???(categorical)**|??????<br>????????……|???????……|


* ??????????? --(???)--> ??????
* pandas.read_csv() ??? dataframe ?column??(df.dtypes)????????:
    * int,float —— ???(?????)
    * object    —— ???(??)
        * ?? column ? object?,???????? 'str'?'float'(NA??float)


In [ ]:
# ?????
feats_numeric  = df_allX.dtypes[df_allX.dtypes != "object"].index.values
#feats_numeric = [attr for attr in df_allX.columns if df_allX.dtypes[attr] != 'object']

# ?????
feats_object = df_allX.dtypes[df_allX.dtypes == "object"].index.values
#feats_object = [attr for attr in df_allX.columns if df_allX.dtypes[attr] == 'object']
#feats_object = df_train.select_dtypes(include = ["object"]).columns

print(feats_numeric.shape,feats_object.shape)

??79???,pandas????? 36????,43????? —— ??????????

In [ ]:
# ??????,??????
feats_numeric_discrete  = ['MSSubClass','OverallQual','OverallCond'] # ???????????????? —— ???????????
feats_numeric_discrete += ['TotRmsAbvGrd','KitchenAbvGr','BedroomAbvGr','GarageCars','Fireplaces'] # ????
feats_numeric_discrete += ['FullBath','HalfBath','BsmtHalfBath','BsmtFullBath'] # ???????????????
feats_numeric_discrete += ['MoSold','YrSold'] # ???,????????????

# ?????
feats_continu = feats_numeric.copy()
# ?????
feats_discrete = feats_object.copy()

for f in feats_numeric_discrete:
    feats_continu = np.delete(feats_continu,np.where(feats_continu == f))
    feats_discrete = np.append(feats_discrete,f)

print(feats_continu.shape,feats_discrete.shape)

????,?????2???:  22??????,57??????

## ????(scatter)

In [ ]:
def plotfeats(frame,feats,kind,cols=4):
    """???????
    
    Parameters
    ----------
    frame : pandas.DataFrame
        ??????
    
    feats : list ? numpy.array
        ???????
        
    kind : str
        ????:'hist'-???;'scatter'-???;'hs'-???????????;'box'-???,??feat???;'boxp'-Price???,feat????????
        
    cols : int
        ???????
    
    Returns
    -------
    None
    """
    rows = int(np.ceil((len(feats))/cols))
    if rows==1 and len(feats)<cols:
        cols = len(feats)
    #print("??%d???,?%d??%d???" % (len(feats), rows, cols))
    if kind == 'hs': #hs:hist and scatter
        fig, axes = plt.subplots(nrows=rows*2,ncols=cols,figsize=(cols*5,rows*10))
    else:
        fig, axes = plt.subplots(nrows=rows,ncols=cols,figsize=(cols*5,rows*5))
        if rows==1 and cols==1:
            axes = np.array([axes])
        axes = axes.reshape(rows,cols) # ? rows=1 ?,axes.shape:(cols,),??reshape??
    i=0
    for f in feats:
        #print(int(i/cols),i%cols)
        if kind == 'hist':
            #frame.hist(f,bins=100,ax=axes[int(i/cols),i%cols])
            frame.plot.hist(y=f,bins=100,ax=axes[int(i/cols),i%cols])
        elif kind == 'scatter':
            frame.plot.scatter(x=f,y='SalePrice',ylim=(0,800000), ax=axes[int(i/cols),i%cols])
        elif kind == 'hs':
            frame.plot.hist(y=f,bins=100,ax=axes[int(i/cols)*2,i%cols])
            frame.plot.scatter(x=f,y='SalePrice',ylim=(0,800000), ax=axes[int(i/cols)*2+1,i%cols])
        elif kind == 'box':
            frame.plot.box(y=f,ax=axes[int(i/cols),i%cols])
        elif kind == 'boxp':
            sns.boxplot(x=f,y='SalePrice', data=frame, ax=axes[int(i/cols),i%cols])
        i += 1
    plt.show()

In [ ]:
plotfeats(df_train,feats_continu,kind='scatter',cols=6)

????:

- LotFrontage?LotArea?GrLivArea?1stFlrSF?2stFlrSF?GarageArea?BsmtFinSF1?TotalBsmtSF: ???????????????????
    - LotFrontage:????????,???50-100??(15-30?),???????????????????……
    - LotArea:????(???????????……),???10516????(900+???),???……
    - GrLivArea:????????
    - 1stFlrSF?2stFlrSF: ?1? 2?????
    - GarageArea:????
    - BsmtFinSF1?BsmtFinSF2?TotalBsmtSF:?????,???????2????
- YearBuilt?YearRemodAdd?GarageYrBlt:???????,???????????,?????,????????????,??????????

In [ ]:
plotfeats(df_train,feats_numeric_discrete,kind='scatter',cols=6)

????:

- MSSubClass:??,????????,?????????????????,???????????
    - 20/30/40: ??(1-STORY)??,?????? storey,????? story
    - 45/50:1-1/2 story,???????,??
    - ……
- OverallQual?OverallCond:?????????condition?????? —— ?????????,?????????????,?????????
- TotRmsAbvGrd?GarageCars:?????????,??????????
- YrSold?MoSold:??????,??06?~10????,???????????,?????????????????,???????
- ?????????????

# **????**

**knowing your data is the most difficult thing in data science**

????????,?????????????????,??????????

??????:

- ?????:?? hist(???)? scatter(???)??
- ?????:?? box(???)??
- ??????:
- ????: ?? bar(???)??
- ?????:?? heatmap(??)??

???? matplotlib ? seaborn,????????????

## ?????

???????????:

* ??????
* ?????????(Shapiro-Wilktest)
* ?????????
* ??-?????? —— ????

### ?????(hist|scatter)

**skewness:??????????**

* ????????????????,??????????????????
* ?????????????????????????
* ????????(?:???)?????,??????(?????)??

In [ ]:
# SalePrice ????
df_train.skew()['SalePrice']

In [ ]:
#df_train.plot(kind='hist',y='SalePrice',bins=100)
df_train['SalePrice'].plot(kind='hist',y='SalePrice',bins=100) # ????????????????
#sns.distplot(df_train['SalePrice'], fit='norm');
#plt.hist(df_train['SalePrice'],bins=100)

????:

????????,????????????,??????????????,?????????????,??????????????
—— ?????,???,???????

In [ ]:
stats.probplot(df_train['SalePrice'], plot=plt)

??? log ????,???????????????,??????? log ????????????????

In [ ]:
df_train['SalePrice'].apply(lambda x: np.log1p(x)).plot(kind='hist',y='SalePrice',bins=100)

In [ ]:
# ??????????
skewed = df_allX[feats_numeric].apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
#skewed = df_allX[feats_numeric].skew().sort_values(ascending=False)
skewed[:10]

In [ ]:
# ????????(SalePrice??)???????
plotfeats(df_train,skewed[:6].index,kind='hs',cols=6)

### ????(hist|scatter)

* kurtosis,peakedness —— ????????
* ????????????????????????
* ????????????????,?????????
* ?????,???????,??????????
* ??????????????
* ??????,????????3??,?????????0?

In [ ]:
df_train.kurt()['SalePrice']

In [ ]:
# ?????????
kurted = df_allX[feats_numeric].kurt().sort_values(ascending=False)
kurted[:10]

In [ ]:
# ????????(SalePrice??)??????
plotfeats(df_train,kurted[:6].index,kind='hs',cols=6)

## ?????(box)

### ???????

In [ ]:
plotfeats(df_train,feats_numeric,kind='box',cols=6)

In [ ]:
# ???????,???????,???????????,???????????

plt.figure(figsize=(16,10))

plt.subplot(121)
sns.boxplot(data=df_allX[feats_continu],orient="h")

plt.subplot(122)
sns.boxplot(data=df_allX[feats_discrete],orient="h")

### SalePrice ????

In [ ]:
plotfeats(df_train, ['OverallQual'], kind='boxp', cols=6)

In [ ]:
plotfeats(df_train, feats_numeric_discrete, kind='boxp', cols=6)

????:

???????????????,???????????????,??????????:????????????,???????????

???????:?????????????,?????(???????),?????????????????,???????????????????,??????????

In [ ]:
plotfeats(df_train, feats_object, kind='boxp', cols=6)

## ??????

Homoscedasticity,????,???????

?????:

* Hartley??
* Bartlett??
* ???Bartlett??

[??????????????](https://www.sohu.com/a/197715883_655370)

## ????(bar)

Analysis of Variance,??ANOVA,??“?????”?“F??”.

**????:**

* ??
    * ????(?????????)???
        * ???????
        * ???????
        * ???????
    * ????(???????F???)???
        * ??????(ANOVA - Analysis Of Variance)
        * ?????(ANCOVA - Analysis of Covariance)
        * ??????(MANOVA - Multivariate Analysis Of Variance)
    * ??
        * ANOVA:
            * ??:
                * ???????????????????
                * ??????????????
                * ??????????
                * ????????????? —— ??
                    * ??:???
                    * ???:????:1?2??2?4?2?……
                    * ??:???
                    * ??????????-???????,????????????????(??)???????,???????????????,????“??”?“??”……???
                * ??????????????? —— ??
            * ??A:
                * ???????????????????
* ??
    * SS(Sum of Squares,???,=?(Yi-Ymean)^2 )????
        * SSE: ?????????????
        * SSF:???????????????
        * SST = SSE + SSF
    * df:???
        * dfE:?? n - m  // m?,??n???
        * dfF:?? m - 1
        * dfT:?? n - 1
    * Mean Square:??? = ??/???
        * ?????: MSE = SSE/dfE = SSE/(n-m)
        * ?????:MSF = SSF/dfF = SSF/(m-1)
        * ?????:MST = SST/dfT = SST/(n-1)
    * F:= MSF/MSE,?????????
        * ??,??????????????
        * ????????,? P ?
    * P(Sig) = F/F????:??????????
        * **P < 0.05 :(????)??**
        * **P = 0 : ???????**
* ??:
    * Matlab:`anova1(x)` 
    * Scipy: `f,p = scipy.stats.f_oneway(sample1, sample2, ...)`

**========= Test =========**

In [ ]:
a = np.random.random(size=(1000,))
b = np.random.random(size=1000,)
f,p = stats.f_oneway(a, b)
print(f,p)

In [ ]:
a = np.random.randn(1000,)
b = np.random.randn(1000,)
f,p = stats.f_oneway(a, b)
print(f,p)

In [ ]:
a = np.random.randint(1,10,size=1000,)
b = np.random.randint(1,10,size=1000,)
f,p = stats.f_oneway(a, b)
print(f,p)

In [ ]:
a = np.random.randint(1,10,size=1000,)
b = np.random.randint(5,15,size=1000,)
f,p = stats.f_oneway(a, b)
print(f,p)

In [ ]:
a = np.random.binomial(5,0.2,size=1000)
b = np.random.randn(1000,)
f,p = stats.f_oneway(a, b)
print(f,p)

**========= Test Over =========**

### scipy.stats.f_oneway()

In [ ]:
# stats.f_oneway() ??????????array 
# ???2???(???X????Y)?dataframe???????
def anovaXY(data):
    samples = []
    X = data.columns[0]
    Y = data.columns[1]
    for level in data[X].unique():
        if (type(level) == float): # np.NaN ?????
            s = data[data[X].isnull()][Y].values
        else:
            s = data[data[X] == level][Y].values
        samples.append(s)
    f,p = stats.f_oneway(*samples) # ??????
    return (f,p)

**????????feature?SalePrice???????????:**

In [ ]:
df = pd.DataFrame(columns=('feature','f','p','logp'))
df['feature'] = feats_discrete
for fe in feats_discrete:
    data = pd.concat([df_train[fe],df_train['SalePrice']],axis=1)
    f,p = anovaXY(data)
    df.loc[df[df.feature==fe].index,'f'] = f
    df.loc[df[df.feature==fe].index,'p'] = p
    df.loc[df[df.feature==fe].index,'logp'] = 1000 if (p==0) else np.log(1./p)

# OverallQual ? p=0,?????????????

In [ ]:
plt.figure(figsize=(10,4))
sns.barplot(data=df.sort_values('p'), x='feature', y='logp')
plt.xticks(rotation=90)

### pandas.Series.corr()

In [ ]:
def spearman(frame, features):
    '''
    ??“????????”????????????(?????)
    '''
    spr = pd.DataFrame()
    spr['feature'] = features
    spr['corr'] = [frame[f].corr(frame['SalePrice'], 'spearman') for f in features] # ????? Series.corr() 
    spr = spr.sort_values('corr')
    plt.figure(figsize=(6, 0.2*len(features)))
    sns.barplot(data=spr, y='feature', x='corr', orient='h')    
    plt.show()

In [ ]:
spearman(df_train, np.delete(df_train.columns.values,-1))

**?????**

* GarageCars ? GarageArea:?? GarageCars ? SalePrice ??????,??????,?? GarageArea
* ??,TotalBsmtSF ??,?? 1stFlrSF

## ?????(-1~+1)

**???????,???????????????????**

?? DataFrame.corr(method='pearson', min_periods=1) ???????,????????? DataFrame,???? heatmap ??????

method ?3?:  
* pearson : standard correlation coefficient —— ??????
* kendall : Kendall Tau correlation coefficient —— KT????
* spearman : Spearman rank correlation —— ????????

???: ???????feature???????

* ? cov(X, Y)>0?,?? X?Y ???;
* ? cov(X, Y)<0?,??X?Y???;
* ? cov(X, Y)=0?,??X?Y????

In [ ]:
corr_pearson = df_train.corr(method='pearson')
corr_spearman = df_train.corr(method='spearman')

In [ ]:
# corrmat ? 38*38???:???? numeric ? feature ??????
corr_pearson.shape

In [ ]:
corr_spearman.shape

### ?????(heatmap)

* heatmap: 
    * matplotlib ?heatmap????,??????
    * sns.heatmap() ???????
* ????,?????,?????????;
* ????,???;

In [ ]:
# ?????figsize,?????????????
# ??????????????sns.heatmap() —— ????bug?
plt.figure(figsize=(20, 20))
plt.subplot(211)
sns.heatmap(corr_pearson, vmax=.8, square=True);
plt.subplot(212)
sns.heatmap(corr_spearman, vmax=.8, square=True);

????:  

* ??2????????:
    * TotalBsmtSF--1stFlrSF:??????1?????? —— ??????,??
    * GarageCars--GarageArea:???????????? —— ????
    * ??4?????????2?,??2????feature
    * ??:**?????**


### ???????(pairplot)

In [ ]:
feats_d = corr_pearson.nlargest(8,'SalePrice').index
feats_d

In [ ]:
sns.pairplot(df_train[feats_d],size=2.5)

????????,??????,??????????

# **????**

## ??????

### ??????

???????,???????????:

* YrSold?MoSold:???????????

??????????????:

* OverallQual: ?????,?????,????????????(??????)
* YearBuild: ???????:????????,??????

In [ ]:
feats_del = ['YrSold','MoSold']

In [ ]:
df_allX.shape

In [ ]:
df_allX.drop(feats_del, axis=1, inplace=True)  # ????? :)

In [ ]:
# ??????4??????
for f in feats_del:
    feats_numeric  = np.delete(feats_numeric,  np.where(feats_numeric  == f))
    feats_object   = np.delete(feats_object,   np.where(feats_object   == f))
    feats_continu  = np.delete(feats_continu,  np.where(feats_continu  == f))
    feats_discrete = np.delete(feats_discrete, np.where(feats_discrete == f))

In [ ]:
df_allX.shape

### ?????

?????????,??????????,?????????????,??????????,????????,????????“???”???????

In [ ]:
# ?????????,??????????feature?????
feats_away = ['LotFrontage','LotArea','BsmtFinSF1','BsmtFinSF2','1stFlrSF','GrLivArea','TotalBsmtSF']
plotfeats(df_train,feats_away,kind='scatter')

**???????** —— ???????!  
??????? SalePrice ????,? df_allX ??? SalePrice,?? df_train ???  
df_allX ?????????Id,??????? Id ???? df_train ???,?????

In [ ]:
ids = []

In [ ]:
df_train.sort_values(by = 'GrLivArea', ascending = False)[:2][['Id','GrLivArea','SalePrice']]
# 1299 ? df_train ?? Id ?, 1298 ?? df_allX ?? index

In [ ]:
ids.append(1299)
ids.append(524)

In [ ]:
df_train.sort_values(by = 'TotalBsmtSF', ascending = False)[:2][['Id','TotalBsmtSF','SalePrice']]
# 1299 ????,???????????????????

In [ ]:
df_train.sort_values(by = '1stFlrSF', ascending = False)[:2][['Id','1stFlrSF','SalePrice']]

In [ ]:
df_train.sort_values(by = 'BsmtFinSF1', ascending = False)[:2][['Id','BsmtFinSF1','SalePrice']]
# ???? 1299,????????

In [ ]:
df_train.sort_values(by = 'LotArea', ascending = False)[:3][['Id','LotArea','SalePrice']]

In [ ]:
ids.append(314)
ids.append(335)
ids.append(250)

In [ ]:
df_train.sort_values(by = 'LotFrontage', ascending = False)[:3][['Id','LotFrontage','SalePrice']]

In [ ]:
ids.append(1299)
ids.append(935)

In [ ]:
np.unique(ids)

In [ ]:
print(df_train.shape,df_test.shape,df_allX.shape)

In [ ]:
for id in np.unique(ids):
    df_train = df_train.drop(df_train[df_train.Id==id].index)
    df_allX = df_allX.drop(df_allX[df_allX.index==(id-1)].index)

In [ ]:
print(df_train.shape,df_test.shape,df_allX.shape)

## ?????

* **??????NA**
    - ??????? 'NA' ?2?:
        1. NA ??????,????/???????????,?????????
        2. NA ????
    - ??????? np.NaN ????????
* ??????,?????????? —— ????
* ??????,????
    - ?????:?min,max,mean,median?mod??
    - ?????:——????

### NaN?NA?????

In [ ]:
# Python ? NaN ???:
print(type(None),type(np.NaN))

????????,?????? np.NaN ? 'NA'

In [ ]:
def NaNRatio(frame,feats):    
    """
    ????? numpy.NaN ??, feats ?????? or ?????
    """
    na_count = frame[feats].isnull().sum().sort_values(ascending=False)
    na_rate = na_count / len(frame)
    na_data = pd.concat([na_count,na_rate],axis=1,keys=['count','ratio'])
    return na_data[na_data['count']>0]

In [ ]:
def NARatio(frame,feats):
    """
    ???????? NA ??
    """
    nadict={}
    for c in feats:        
        # ??1:
        # frame[f][frame[f]=='NA'] —— ??????????? object ?,numeric????        
        # ??2:
        for r in frame.index:
            if 'NA'==frame.loc[r,c]:
                if 0==nadict.get(c,0):
                    nadict[c]=[]
                nadict[c].append(r)
    return nadict

In [ ]:
def transNaNtoNumber(frame, column, method, val=0):
    """
    ? numpy.NaN ???????
    """
    if method == 'mean':
        frame[column] = frame[column].fillna(round(frame[column].mean()))
    elif method == 'min':
        frame[column] = frame[column].fillna(round(frame[column].min()))
    elif method == 'max':
        frame[column] = frame[column].fillna(round(frame[column].max()))
    elif method == 'special':
        frame[column] = frame[column].fillna(val).round()
    else:
        return
    return frame

In [ ]:
def transNaNtoNA(frame, feature):
    """
    ? numpy.NaN ????? NA
    """
    # frame[feature][df[feature].isnull()] = 'NA' # ????warnning
    frame.loc[frame[feature].isnull(),feature] = 'NA'

In [ ]:
def transNAtoNumber(frame,feat,val=0):
    """
    ???? NA ???????(??0)
    """
    for r in frame[frame[feat]=='NA'].index:
        frame.loc[r,feat] = val
    return frame

### ???:min,max,mean

In [ ]:
# ???????? NA ???????
pd.concat([NaNRatio(df_train,feats_numeric),NaNRatio(df_test,feats_numeric)],axis=1,sort=True)

????????,????????????,??????????????,????????mean,?????,??????,????????

In [ ]:
#LotFrontage:??????:????
df_allX = transNaNtoNumber(df_allX,'LotFrontage','mean') 

#GarageYrBlt:???????:???????????,????????,?????
df_allX = transNaNtoNumber(df_allX,'GarageYrBlt','min')

#MasVnrArea:??????,??????????0,??????
df_allX = transNaNtoNumber(df_allX,'MasVnrArea','special',0)

#??:????,???mean??
df_allX = df_allX.fillna(df_allX.mean())

In [ ]:
# df_allX ??????????? np.NaN
pd.concat([NaNRatio(df_allX,feats_numeric)],axis=1)

### ??? -- ??????

???float???NaN???char?"NA",????'NA'??????,????,??????????

In [ ]:
pd.concat([NaNRatio(df_train,feats_object),NaNRatio(df_test,feats_object)],axis=1,sort=True)

In [ ]:
for c in feats_object:
    transNaNtoNA(df_allX,c)

In [ ]:
# df_allX ??????????? np.NaN
NaNRatio(df_allX,feats_object)

## ???(Normalization)??

??? (normalization)  
????????????????(??? -1 ? +1 ? 0 ? 1)??????,???????????? 800 ? 6000???????
???,????????????? -1 ? +1 ????  
—— Google ??????-???(Machine Learning Glossary_Google Developers)

???????:

1. z-score
$$x_i = \frac{x_i - \mathbb{E} x_i}{\text{std}(x_i)}?$$
2. Min-Max Scaling
$$x_i = \frac{x_i - min}{max - min}$$
3. Decimal scaling???????

In [ ]:
df_allX[feats_numeric] = df_allX[feats_numeric].apply(lambda x:(x-x.mean())/(x.std()))

????????

In [ ]:
plt.figure(figsize=(16,10))

plt.subplot(121)
sns.boxplot(data=df_allX[feats_continu],orient="h")

plt.subplot(122)
sns.boxplot(data=df_allX[feats_discrete],orient="h")

## ?????

* **??/????????????(encoding):**
    * **???-??????:?????????**
    * **???-??????:???????**,????????????????,?OverallQual,10~1?????VeryExcellent~VeryPoor

???????????:

1. One-Hot Encoding: ?????????? —— ?????????,so,??? df_allX ???
2. ??-??-???:??????????,?????y???,????,???????????1,2,3... 

????2?1,???????1,???2,???1??,??1???1???,2??????,????????????,???

?????????,?????,1????????,??????,?2????

### One-Hot Encoding

??:????(??????low?),???????,???1bit???????

??:

* ?????????????????,????????,??????,??????????????……??
* ??????????????
* ?????????????????

??:

1. 
```
from sklearn import preprocessing
preprocessing.OneHotEncoder(sparse=False).fit_transform()
```
2. 
```
pandas.get_dummies() //dummy variables(???????????????)
```

In [ ]:
df_allX.shape

In [ ]:
#df_allX = pd.get_dummies(df_allX[feats_object], dummy_na=True)

In [ ]:
df_allX.shape

### ??-??-?????

* **?SalePrice???,?????????????SalePrice?????,?????1? 2? 3……???**
* **????SalePrice????,???? df_train ??,????? df_allX ?**
* **????: np.NaN ? 'NA' ??????**
* **??:??????? df_train ???NA,?? df_test ??NA,?????**

#### ?SalePrice??????

In [ ]:
def encode(frame, feature, targetfeature='SalePrice'):
    ordering = pd.DataFrame()
    # ??????????,????df???
    ordering['val'] = frame[feature].unique()
    ordering.index = ordering.val
    # ??????,?????????
    ordering['price_mean'] = frame[[feature, targetfeature]].groupby(feature).mean()[targetfeature]
    # ????order???1?2?3?……
    ordering = ordering.sort_values('price_mean')
    ordering['order'] = range(1, ordering.shape[0]+1)
    ordering = ordering['order'].to_dict()
    return ordering

**??????'NA'??? encode(),????? groupby() ???? NaN ? **

??:

In [ ]:
encode(df_train,'BsmtCond') # numpy.NaN ?????????

In [ ]:
dfc = df_train.copy()
transNaNtoNA(dfc,'BsmtCond') # ? NaN ?? 'NA'
encode(dfc,'BsmtCond') # ??? NA ??????

????

In [ ]:
# ????
df_allX.loc[20:30,'Alley'] 

In [ ]:
dfc = df_train.copy()

for fb in feats_object:
    print("\r\n-----\r\n",fb,end=':')
    transNaNtoNA(dfc,fb)
    for attr_v, score in encode(dfc,fb).items():
        print(attr_v,score,end='\t')
        df_allX.loc[df_allX[fb] == attr_v, fb] = score        

In [ ]:
# ????
df_allX.loc[20:30,'Alley'] 

#### ?????????????

???? train ??? NA,test ?? NA,??????

In [ ]:
# ???????? numpy.NaN
NaNRatio(df_allX,df_allX.columns.values)

In [ ]:
# ???????? 'NA'
stillNA = NARatio(df_allX,df_allX.columns.values)
stillNA

In [ ]:
df_allX.loc[1914:1916,'MSZoning'] #???

??,??????:

MSZoning: ????:??????????????????????  
???????? OverallQual(???????????????)?:  
2? 1? 5? 1 —— ????  

**??????????**

In [ ]:
df_allX[['MSZoning','OverallQual']][df_allX['MSZoning']=='NA']

In [ ]:
dftemp = df_allX.copy()
for sn in stillNA.keys():
    dftemp  = transNAtoNumber(dftemp,sn)
    df_allX = transNAtoNumber(df_allX,sn,dftemp[sn].mean())

In [ ]:
df_allX.loc[1914:1916,'MSZoning']

In [ ]:
NARatio(df_allX,df_allX.columns.values)

?????????,????????????????,????????   

????????,??????????????????,??????????????,????????2???,?????????????,????????????,????????,?????  

?????,????Python ?????????,?????????????,???:????????????

??2??????,???????:???????RMSE????0.02????,??ML??,??0.05~0.1??,????????????ML????????

???????,??????? -- ????????

# **????**

**RMSE**: Root-Mean-Squared-Error,??????,[wikipedia](https://en.wikipedia.org/wiki/Root-mean-square_deviation) ,??????????????


In [ ]:
def get_rmse_log(net, X_train, y_train):
    num_train = X_train.shape[0]
    clipped_preds = nd.clip(net(X_train),1,float('inf'))
    return np.sqrt( 2 * nd.sum(square_loss(nd.log(clipped_preds), nd.log(y_train))).asscalar()/num_train)

## **??**

### MXNet

In [ ]:
import mxnet
print(mxnet.__version__)

In [ ]:
from mxnet import nd, autograd, gluon

???? mxnet ??? array ?????,??? GPU ????

In [ ]:
num_train = df_train.shape[0]
X_train = nd.array(df_allX[:num_train])
X_test  = nd.array(df_allX[num_train:].values)
y_train = nd.array(df_train.SalePrice.values)

In [ ]:
square_loss = gluon.loss.L2Loss()

In [ ]:
def get_net(units=128, dropout=0.1):
    net = gluon.nn.Sequential()
    with net.name_scope():  
        if units != 0:
            net.add(gluon.nn.Dense(units, activation='relu'))
        if dropout != 0:
            net.add(gluon.nn.Dropout(dropout))
        net.add(gluon.nn.Dense(1))
    net.initialize()
    return net

In [ ]:
def train(net, X_train, y_train, X_test, y_test, epochs, learning_rate, weight_decay):
    train_loss = []
    if X_test is not None:
        test_loss = []
    batch_size = 100
    dataset_train = gluon.data.ArrayDataset(X_train, y_train)
    data_iter_train = gluon.data.DataLoader(dataset_train, batch_size,shuffle=True)
    trainer = gluon.Trainer(net.collect_params(), 'adam',
                            {'learning_rate': learning_rate,
                             'wd': weight_decay})
    net.collect_params().initialize(force_reinit=True)
    for epoch in range(epochs):
        for data, label in data_iter_train:
            with autograd.record():
                output = net(data)
                loss = square_loss(output, label) 
            loss.backward()
            trainer.step(batch_size)

        # ??? L2Loss,?????? RMSE Loss
        train_loss.append(get_rmse_log(net, X_train, y_train))
        if X_test is not None:
            test_loss.append(get_rmse_log(net, X_test, y_test))

        
    # ???? epochs ??? loss
    if X_test is not None:
        return train_loss, test_loss
    else:
        return train_loss

### pytorch

In [ ]:
import torch
print(torch.__version__)

In [ ]:
from torch import nn, autograd as ag, optim

In [ ]:
# ???

### TensorFlow

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
# ???

### PaddlePaddle

In [ ]:
# ???

## **??**

????????? MXNet Demo ?,??? K-?????,??????:

- Andrew Ng ?????3?:?????????????????????????????,?????????,?????????????????SalePrice,?????
- K-?????:? ??? ?k?,???1??????,?? (k-1) ?????

??????? savejpg ??,?????kaggle???,?????????????

In [ ]:
def k_fold_cross_valid(k, epochs, X_train, y_train, learning_rate, weight_decay, units=128, dropout=0.1, savejpg=False):
    assert k > 1
    fold_size = X_train.shape[0] // k
    train_loss_sum = 0.0
    test_loss_sum = 0.0
    train_loss_std_sum = 0.0
    test_loss_std_sum = 0.0

    cols = k
    rows = int(np.ceil(k/cols))
    fig, axes = plt.subplots(nrows=rows,ncols=cols,figsize=(cols*5,rows*5))
        
    for test_i in range(k):
        X_val_test = X_train[test_i * fold_size: (test_i + 1) * fold_size, :]
        y_val_test = y_train[test_i * fold_size: (test_i + 1) * fold_size]

        val_train_defined = False
        for i in range(k):
            if i != test_i:
                X_cur_fold = X_train[i * fold_size: (i + 1) * fold_size, :]
                y_cur_fold = y_train[i * fold_size: (i + 1) * fold_size]
                if not val_train_defined:
                    X_val_train = X_cur_fold
                    y_val_train = y_cur_fold
                    val_train_defined = True
                else:
                    X_val_train = nd.concat(X_val_train, X_cur_fold, dim=0)
                    y_val_train = nd.concat(y_val_train, y_cur_fold, dim=0)
        
        net = get_net(units=units, dropout=dropout)
        train_loss, test_loss = train(
            net, X_val_train, y_val_train, X_val_test, y_val_test, 
            epochs, learning_rate, weight_decay)        
        print("%d-fold \tTrain loss:%f \tTest loss: %f" % (test_i+1, train_loss[-1], test_loss[-1]))
        
        axes[test_i%cols].plot(train_loss, label='train')
        axes[test_i%cols].plot(test_loss, label='test')
        
        train_loss_sum += np.mean(train_loss[-10:])
        test_loss_sum += np.mean(test_loss[-10:])
        
        train_loss_std_sum += np.std(train_loss[10:])
        test_loss_std_sum  += np.std(test_loss[10:])
    
    print("%d-fold Avg: train loss: %f, Avg test loss: %f, Avg train lost std: %f, Avg test lost std: %f" % 
          (k, train_loss_sum/k, test_loss_sum/k, train_loss_std_sum/k, test_loss_std_sum/k))

    if savejpg:
        #plt.savefig("~/house-prices/%d-%d-%.3f-%d-%d-%.3f.jpg" %(k,epochs,learning_rate,weight_decay,units,dropout))
        plt.close()
    else:
        plt.show()
        
    return train_loss_sum / k, test_loss_sum / k, train_loss_std_sum / k, test_loss_std_sum /k

In [ ]:
# ?????????????,?????????????

k=5
epochs=50
learning_rate=5
weight_decay=0
units=0
dropout=0

train_avg_loss, test_avg_loss, train_avg_loss_std, test_avg_loss_std = k_fold_cross_valid(
    k, epochs, X_train, y_train, learning_rate, weight_decay, units, dropout, savejpg=False)

## **??**

??????:???6???,????????,?????????,????????

In [ ]:
# ????
def expand(mulcoldf, sigcoldf):
    r = pd.DataFrame(columns=np.append(mulcoldf.columns.values, sigcoldf.columns.values))
    for x in sigcoldf.values:
        s = mulcoldf.copy()
        s[sigcoldf.columns[0]] = x[0]
        r = pd.concat([r,s])
    return r

# k,epochs,learning_rate,weight_decay,units,dropout
def get_params(k=[5],epochs=[50],learning_rate=[5,0.5],weight_decay=[0],units=[0,128],dropout=[0,0.01]):
    p = pd.DataFrame()
    p = expand(pd.DataFrame({'k':k}), pd.DataFrame({'epochs':epochs}))
    p = expand(p, pd.DataFrame({'learning_rate':learning_rate}))
    p = expand(p, pd.DataFrame({'weight_decay':weight_decay}))
    p = expand(p, pd.DataFrame({'units':units}))
    p = expand(p, pd.DataFrame({'dropout':dropout}))
    return p.reset_index(drop=True)

In [ ]:
params = get_params(learning_rate=[0.1,0.5,1,2,3,4,5], weight_decay=[1,10,100,130,150,500], units=[64,128,256])
params[-5:]

????  
????????  
????????  
????????  
????????  

In [ ]:
# ?????????,????????? kaggle ??
params = params[:8]

In [ ]:
dfrult = pd.DataFrame(columns=('k','epochs','learning_rate','weight_decay','units','dropout','train_avg_loss','test_avg_loss','train_avg_loss_std','test_avg_loss_std'))
i = 0
for param in params.values:
    print("%s %d" % ("="*80,i))
    i += 1
    k,epochs,learning_rate,weight_decay,units,dropout = param.tolist()    
    print("k-fold=%d,epochs=%d,learning_rate=%f,weight_decay=%f,units=%d,dropout=%d" % (k,epochs,learning_rate,weight_decay,units,dropout))
    
    train_avg_loss, test_avg_loss, train_avg_loss_std, test_avg_loss_std = k_fold_cross_valid(k, epochs, X_train, y_train, learning_rate, weight_decay, units, dropout, savejpg=True)
    
    temp = pd.DataFrame([[k,epochs,learning_rate,weight_decay,units,dropout,train_avg_loss,test_avg_loss,train_avg_loss_std,test_avg_loss_std]],
                columns=['k','epochs','learning_rate','weight_decay','units','dropout','train_avg_loss','test_avg_loss','train_avg_loss_std','test_avg_loss_std'])
    dfrult = pd.concat([dfrult, temp])

savejpg=True ??????????,???????????????????,?????????????,????????,?????k??(k-??k)?

????????????:

In [ ]:
df = dfrult.copy()

In [ ]:
df[:5]

In [ ]:
df['diff'] = df['test_avg_loss'] - df['train_avg_loss']
df['sum'] = df['test_avg_loss'] + df['train_avg_loss']
df = df.sort_values('sum').reset_index(drop=True)
df[:5]

In [ ]:
# ??
df.loc[df['learning_rate']==0.1].loc[df['weight_decay']==130].loc[df['units']==128].loc[df['dropout']==0.01]

In [ ]:
df[df['diff']<0.03][:5]

## **??**

?? test ???????????????

In [ ]:
def learn(epochs, X_train, y_train, test, learning_rate, weight_decay, units, dropout):
    net = get_net(units=units, dropout=dropout)
    train_loss = train(net, X_train, y_train, None, None, epochs, learning_rate, weight_decay)
    plt.plot(train_loss)
    plt.show()
    print("train loss last 10 data avg: %f" % np.mean(train_loss[-10:]))
    preds = net(X_test).asnumpy()
    test['SalePrice'] = pd.Series(preds.reshape(1, -1)[0])
    submission = pd.concat([test['Id'], test['SalePrice']], axis=1)
    submission.to_csv('submission.csv', index=False)

In [ ]:
# ??????
#k,epochs,learning_rate,weight_decay,units,dropout = (5,50,0.1,130,128,0.01)
k,epochs,learning_rate,weight_decay,units,dropout = df.iloc[0][0:6]
print("k-fold=%d,epochs=%d,learning_rate=%f,weight_decay=%f,units=%d,dropout=%d" % (k,epochs,learning_rate,weight_decay,units,dropout))

learn(epochs, X_train, y_train, df_test, learning_rate, weight_decay, units, dropout)

????,??? kaggle ???????: submission.csv

???????: `kaggle c submit -c house-prices-advanced-regression-techniques -f submission.csv -m "I love kaggle"` 

??????,?????100??

Good luck!